In [79]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import pandas as pd
import os
from math import sqrt, pow, exp
from spacy import *

os.environ["TRANSFORMERS_NO_TF"] = "1"

from sentence_transformers import SentenceTransformer

RuntimeError: Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

In [56]:
nltk.download("wordnet")
nltk.download("omw-1.4")

wnl = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
def preprocess_nltk(text):
    tokens = word_tokenize(text.lower().replace("’", "'"))
    tagged = nltk.pos_tag(tokens)
    lemmatized = [
        wnl.lemmatize(word, get_wordnet_pos(pos))
        for word, pos in tagged
        if word not in string.punctuation
    ]
    return lemmatized

def join_tokens(tokens):
    return " ".join(tokens)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\huber\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\huber\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [57]:
def load_reviews(folder_path):
    reviews = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            with open(os.path.join(folder_path, filename), 'r', encoding='utf-8') as f:
                reviews.append(f.read())
    return reviews

def load_review(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

In [58]:
def jaccard_similarity(x,y):
  """ returns the jaccard similarity between two lists """
  intersection_cardinality = len(set.intersection(*[set(x), set(y)]))
  union_cardinality = len(set.union(*[set(x), set(y)]))
  return intersection_cardinality/float(union_cardinality)
 
def squared_sum(x):
  """ return 3 rounded square rooted value """
 
  return round(sqrt(sum([a*a for a in x])),3)
 
def euclidean_distance(x,y):
  """ return euclidean distance between two lists """
 
  return sqrt(sum(pow(a-b,2) for a, b in zip(x, y)))

def distance_to_similarity(distance):
  return 1/exp(distance)

def cos_similarity(x,y):
  """ return cosine similarity between two lists """
 
  numerator = sum(a*b for a,b in zip(x,y))
  denominator = squared_sum(x)*squared_sum(y)
  return round(numerator/float(denominator),3)

In [59]:
reviews = load_reviews("data/AC Shadows/html texts")
lemmatized_reviews = [preprocess_nltk(r) for r in reviews]
print(lemmatized_reviews[0])

['assassin', "'s", 'creed', 'shadow', 'review', 'struggle', 'under', 'it', 'weight', "ubisoft's", 'assassin', "'s", 'creed', 'shadow', 'bring', 'player', 'to', 'late', '16th-century', 'japan', 'for', 'the', 'franchise', "'s", 'first', 'main', 'series', 'entry', 'set', 'in', 'east', 'asia', 'it', 'first', 'attempt', 'at', 'dual', 'protagonist', 'in', 'a', 'decade', 'and', 'it', 'first', 'new', 'massive', 'open-world', 'rpg', 'title', 'since', 'assassin', "'s", 'creed', 'valhalla', 'debut', 'in', '2020.', 'the', 'first', 'title', 'integrate', 'into', 'ubisoft', "'s", 'new', 'assassin', "'s", 'creed', 'launcher', 'the', 'animus', 'hub', 'assassin', "'s", 'creed', 'shadow', 'split', 'it', 'narrative', 'and', 'it', 'gameplay', 'between', 'two', 'antagonist', 'naoe', 'a', 'shinobi', 'whose', 'home', 'be', 'under', 'threat', 'from', 'the', 'oda', 'clan', 'and', 'yasuke', 'a', 'samurai', 'who', "'s", 'indebted', 'to', 'his', 'lord', 'for', 'free', 'him', 'from', 'portuguese', 'slaver', 'and', 

In [60]:
review_names = os.listdir("data/AC Shadows/html texts")
review_names = [r.replace(".txt", "") for r in review_names]

In [ ]:
nlp = load("en_core_web_lg")
embeddings = [
    nlp(join_tokens(review)).vector
    for review in lemmatized_reviews]

In [63]:
# -------------------------------------------------JACCARD---------------------------------------------------------------------------------
jaccard_df = pd.DataFrame(index=review_names, columns=review_names)

for i in range(len(lemmatized_reviews)):
    for j in range(i, len(lemmatized_reviews)):
        sim = jaccard_similarity(lemmatized_reviews[i], lemmatized_reviews[j])
        if sim > 0.3:
            jaccard_df.iloc[i, j] = sim
        # print(f"Jaccard similiarity for {review_names[i]} and {review_names[j]}: {sim}")

jaccard_df = jaccard_df.astype(float)

In [64]:
jaccard_df.to_excel("jaccard.xlsx")

In [65]:
# -------------------------------------------------EUCLIDAN---------------------------------------------------------------------------------
euclidan_df = pd.DataFrame(index=review_names, columns=review_names)

for i in range(len(embeddings)):
    for j in range(i+1, len(embeddings)):
        sim = distance_to_similarity(euclidean_distance(embeddings[i], embeddings[j]))
        if sim > 0.8:
            euclidan_df.iloc[i, j] = sim
            print(f"Euclidan similiarity for {review_names[i]} and {review_names[j]}: {sim}")

distance = distance_to_similarity(euclidean_distance(embeddings[0], embeddings[1]))
print(distance)

Euclidan similiarity for ButWhyTho and CNET: 0.8156876776075475
Euclidan similiarity for ButWhyTho and New Game Network: 0.8020220889946784
Euclidan similiarity for ButWhyTho and Worthplaying PC: 0.8020183808832089
Euclidan similiarity for Checkpoint and Digital Trends: 0.8233443180344371
Euclidan similiarity for Checkpoint and Finger Guns: 0.8496645415581329
Euclidan similiarity for Checkpoint and Hooked Gamers: 0.8421878137013002
Euclidan similiarity for CNET and digitalchumps: 0.8042470541454291
Euclidan similiarity for CNET and Finger Guns: 0.8073872738522538
Euclidan similiarity for CNET and gamepressure: 0.8115939786495797
Euclidan similiarity for CNET and GameSpot: 0.800054966564327
Euclidan similiarity for CNET and GamingTrend: 0.8248094034850645
Euclidan similiarity for CNET and Inverse: 0.8003087668832494
Euclidan similiarity for CNET and Worthplaying PC: 0.8028910673729699
Euclidan similiarity for COG and ScreenRant: 0.8027655316057272
Euclidan similiarity for Creative Bloq 

In [66]:
euclidan_df.to_excel("euclidan.xlsx")

In [70]:
# -------------------------------------------------COSINE---------------------------------------------------------------------------------
cosine_df = pd.DataFrame(index=review_names, columns=review_names)

for i in range(len(embeddings)):
    for j in range(i+1, len(embeddings)):
        sim = cos_similarity(embeddings[i], embeddings[j])
        if sim < 0.9:
            cosine_df.iloc[i, j] = sim
            print(f"Cosine similiarity for {review_names[i]} and {review_names[j]}: {sim}")

print(cos_similarity(embeddings[0], embeddings[1]))

0.992


In [68]:
cosine_df.to_excel("cosine.xlsx")

In [69]:
print(embeddings[0])

[ 2.48165173e-03  9.92538929e-02 -1.43963382e-01 -8.92307027e-04
  8.28685910e-02 -5.01201265e-02 -4.47274148e-02  9.68375430e-03
  7.13579217e-03  2.21702194e+00 -1.51176125e-01 -2.14486606e-02
  3.03435326e-02 -1.32506937e-02 -4.04024236e-02 -6.76461011e-02
 -6.81523681e-02  1.10521698e+00 -2.02417091e-01 -4.99543510e-02
 -1.12387538e-02 -6.64631799e-02 -4.08326797e-02 -6.23108447e-02
  6.63135527e-03  5.59974611e-02 -1.27761653e-02 -3.92466113e-02
  1.03490651e-02 -5.65615706e-02 -4.47924137e-02  1.04586825e-01
 -7.90599361e-02  6.75139874e-02  4.77987751e-02 -1.08360305e-01
  1.67149827e-02  5.32414317e-02 -3.78078669e-02 -9.45184305e-02
 -1.93406746e-03  1.08814843e-01 -9.31955595e-03 -8.72743577e-02
  7.26206079e-02  1.92772541e-02 -1.21689774e-01  2.05351114e-02
 -5.58575019e-02  3.46416831e-02 -2.59639658e-02  8.93826224e-03
  9.32044117e-04  1.41989496e-02  5.21416590e-02  3.53120565e-02
 -3.56028713e-02 -3.89524363e-02  2.55035027e-03 -5.52766547e-02
  2.55948305e-02  3.55988